# MIVA Open University
## Course: MIVA-DTS 305 - Data Quality and Data Wrangling
### Tutor-Marked Assessment: Question 2 - Web Scraping and Feature Engineering

**Student Details:**
- **Full Name:** Richard John
- **Email:** richard.john@miva.edu.ng
- **Department:** Data Science
- **Matriculation Number:** 2024/C/DSC/0548

---

### Project Overview & Objectives
This lab focuses on automated web data collection, text cleaning, feature engineering, and exploratory visual analysis using python:
1. Programmatically request and parse HTML from the public sandbox website `http://books.toscrape.com/`.
2. Extract key attributes for all books listed on the homepage: `Title`, `Price`, `Rating`, and `Availability`.
3. Store the extracted information in a structured pandas DataFrame and export it to CSV.
4. Clean the data:
   - Convert currency string prices into float numeric values.
   - Map qualitative star-rating words (e.g., 'One', 'Two', 'Three') into ordinal integers (1-5).
5. Engineer new analytical features:
   - Categorise books into `'Affordable'` (< £20) and `'Expensive'` (>= £20).
   - Evaluate the relationship between book affordability and customer satisfaction (ratings).
6. Present the empirical findings using visualisations (Pie chart of affordability proportions, and Bar chart of mean ratings).


In [ ]:
# Step 1: Import necessary libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display and plotting settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)

print("Libraries imported successfully.")


In [ ]:
# Step 2: Fetch the webpage
target_url = "http://books.toscrape.com/"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}

response = requests.get(target_url, headers=headers, timeout=15)
print(f"HTTP Status Code: {response.status_code}")

if response.status_code == 200:
    print("Page fetched successfully. HTML content length:", len(response.text), "characters")
else:
    raise ConnectionError(f"Failed to fetch {target_url}")


In [ ]:
# Step 3: Parse HTML and extract target book attributes
soup = BeautifulSoup(response.content, 'html.parser')

# Find all product containers on the homepage
book_containers = soup.find_all('article', class_='product_pod')
print(f"Total book containers identified: {len(book_containers)}")

scraped_data = []

for pod in book_containers:
    # 1. Title: Stored in the 'title' attribute of the <a> tag inside <h3>
    title = pod.h3.a['title']
    
    # 2. Price: Found inside <p class="price_color">
    price_text = pod.find('p', class_='price_color').text.strip()
    
    # 3. Rating: Encoded as a secondary class on <p class="star-rating [Word]">
    rating_tag = pod.find('p', class_='star-rating')
    rating_class = rating_tag['class'][1] if rating_tag and len(rating_tag['class']) > 1 else 'None'
    
    # 4. Availability: Located inside <p class="instock availability">
    availability = pod.find('p', class_='instock availability').text.strip()
    
    scraped_data.append({
        'Title': title,
        'Price_Raw': price_text,
        'Rating_Text': rating_class,
        'Availability': availability
    })

# Convert list of dictionaries into a pandas DataFrame
df_scraped = pd.DataFrame(scraped_data)
print("Preview of scraped records:")
df_scraped.head(5)


In [ ]:
# Save raw scraped dataset to CSV
df_scraped.to_csv('../data/books_scraped_raw.csv', index=False)
print("Raw scraped data exported to: ../data/books_scraped_raw.csv")


### Step 4: Data Cleaning
The raw scraped data contains formatted strings that cannot be directly analyzed:
1. **Price:** Contains currency symbols (e.g. `£51.77` or encoding artifacts like `Â£51.77`). We strip extraneous characters and cast the column to a 64-bit float.
2. **Rating:** Qualitative string representations (`'One'`, `'Two'`, `'Three'`, `'Four'`, `'Five'`) must be mapped to ordinal integers (1 to 5).


In [ ]:
# 1. Clean Price column
df_cleaned_books = df_scraped.copy()
df_cleaned_books['Price (£)'] = (
    df_cleaned_books['Price_Raw']
    .str.replace('£', '', regex=False)
    .str.replace('Â', '', regex=False)
    .str.strip()
    .astype(float)
)

# 2. Map Rating strings to numerical values
rating_dictionary = {
    'One': 1,
    'Two': 2,
    'Three': 3,
    'Four': 4,
    'Five': 5
}
df_cleaned_books['Rating'] = df_cleaned_books['Rating_Text'].map(rating_dictionary)

print("Data types after cleaning:")
print(df_cleaned_books[['Price (£)', 'Rating']].dtypes)
print("\nFirst 5 cleaned rows:")
df_cleaned_books[['Title', 'Price (£)', 'Rating', 'Availability']].head()


### Step 5: Feature Engineering
We now derive two key analytical features:
1. **Affordability Classification:** Books with a price strictly below £20 are labeled as `'Affordable'`, while those priced at £20 or above are labeled as `'Expensive'`.
2. **Category Summary Statistics:** We calculate aggregate metrics (mean rating, standard deviation, and count) across affordability segments.


In [ ]:
# Engineer 'Affordability' column
df_cleaned_books['Affordability'] = np.where(
    df_cleaned_books['Price (£)'] < 20.0,
    'Affordable',
    'Expensive'
)

# Export finalized dataset
df_cleaned_books.to_csv('../data/books_scraped_cleaned.csv', index=False)
print("Cleaned & feature-engineered dataset saved to: ../data/books_scraped_cleaned.csv")

# Compute statistical summary
affordability_stats = df_cleaned_books.groupby('Affordability').agg(
    Book_Count=('Title', 'count'),
    Mean_Price=('Price (£)', 'mean'),
    Mean_Rating=('Rating', 'mean'),
    Std_Rating=('Rating', 'std')
).round(2).reset_index()

print("\nStatistical Breakdown by Affordability:")
print(affordability_stats)


### Step 6: Visualisations & Insights

#### 1. Proportion of Affordable vs. Expensive Books (Pie Chart)


In [ ]:
# Pie Chart of Affordability Proportions
counts = df_cleaned_books['Affordability'].value_counts()

plt.figure(figsize=(7, 7))
colors = ['#dc2626', '#16a34a'] # Red for Expensive, Green for Affordable
explode = (0.05, 0.05)

plt.pie(
    counts,
    labels=counts.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    explode=explode,
    shadow=True,
    textprops={'fontsize': 12, 'fontweight': 'bold'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)

plt.title('Proportion of Books: Affordable (< £20) vs. Expensive (≥ £20)', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../figures/q2_affordability_pie_chart.png', dpi=300)
plt.show()


#### 2. Average Customer Rating by Affordability (Bar Chart)


In [ ]:
# Bar Chart of Mean Ratings
plt.figure(figsize=(7, 5))
categories = affordability_stats['Affordability']
mean_ratings = affordability_stats['Mean_Rating']
bar_colors = ['#16a34a' if c == 'Affordable' else '#dc2626' for c in categories]

bars = plt.bar(categories, mean_ratings, color=bar_colors, width=0.45, edgecolor='black', linewidth=0.8)

for bar in bars:
    yval = bar.get_height()
    plt.annotate(f"{yval:.2f} / 5.00",
                 xy=(bar.get_x() + bar.get_width() / 2, yval),
                 xytext=(0, 6),
                 textcoords="offset points",
                 ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.title('Average Customer Rating by Affordability Category', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Affordability Category', fontsize=12, labelpad=10)
plt.ylabel('Mean Rating (Stars: 1 to 5)', fontsize=12, labelpad=10)
plt.ylim(0, 5.5)
plt.yticks(range(0, 6))
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../figures/q2_ratings_bar_chart.png', dpi=300)
plt.show()


### Step 7: Summary of Web Scraping & Analytical Insights
1. **Catalog Distribution:** On the homepage, 85.0% (17 out of 20 books) fall into the **Expensive** category ($\ge £20$), while only 15.0% (3 books) are **Affordable** ($< £20$).
2. **Pricing Structure:** The average price for expensive books is **£43.76**, whereas affordable books average **£16.46**.
3. **Rating Correlation:** Affordable books received a higher average customer rating (**3.33 / 5.00**) compared to expensive books (**2.76 / 5.00**). This suggests that customer expectations may be steeper for premium titles, leading to harsher evaluations when perceived value fails to meet the premium price point.
